# Notebook 15 — Explainability and Faithfulness Checks

**Project:** Report-Guided Weak Supervision for Multi-Label Knee MRI Abnormality Detection  
**Main explanatory model:** Notebook 13 confidence-weighted weak-supervision hierarchical DINOv2 MIL.  
**Reference model:** Notebook 12 gold-only hierarchical DINOv2 MIL.  
**Evaluation reference:** Notebook 14.

This notebook follows the project plan's explanation hierarchy: **series importance → 2.5D-instance/slice importance → embedding-feature attribution → perturbation faithfulness**. It recomputes attention from the Notebook 13 fold checkpoints on the matching outer-held-out gold studies, so explanations correspond to the report-guided model. Notebook 12's saved OOF attention outputs are retained as a gold-only reference.

**Interpretation:** attention weights show how the model aggregates instances. They are not clinical explanations or proof of causality. Integrated Gradients are computed in DINOv2 feature space, not as a pixel-level Grad-CAM heatmap. A pixel-region heatmap would require retaining the image encoder computation graph and corresponding preprocessed MRI pixels.

### Attach through Kaggle “Add Input”
1. RSNA competition dataset (`train.csv`).
2. Notebook 09 output containing the **gold-scope** DINOv2 embeddings and metadata.
3. Notebook 12 output containing OOF predictions and OOF slice/series attention CSVs.
4. Notebook 13 output containing `oof_predictions_all_modes.csv` and the five `gold_plus_confidence_weighted_weak_fold_*.pth` checkpoints.
5. Notebook 14 output containing `overall_metrics.csv` (optional reference; the notebook still runs without it).

The notebook searches `/kaggle/input` and `/kaggle/working`, following the same input-discovery pattern as the preceding notebooks. No report text is used as an inference input.

## 1. Imports, configuration and reproducibility

In [ ]:
import os, gc, json, random, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score

warnings.filterwarnings("ignore")
SEED = 42
N_FOLDS = 5
EMBED_DIM = 384
NUM_CLASSES = 12
ATTENTION_DIM = 128
HIDDEN_DIM = 128
DROPOUT = 0.25
THRESHOLD = 0.50
MAX_IG_STUDIES = 8
IG_STEPS = 12
OUTPUT_DIR = Path("/kaggle/working/15_explainability")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
LABEL_COLS = [
    "ACL", "MCL", "Medial Meniscus", "Lateral Meniscus",
    "Medial OA", "Lateral OA", "PF OA", "Effusion",
    "Synovitis", "Baker's", "Contusion", "Fracture"
]

def seed_everything(seed=SEED):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)

seed_everything()
if torch.get_num_threads() > 4: torch.set_num_threads(4)
print("Device:", DEVICE, "| Output:", OUTPUT_DIR)

## 2. Locate the dataset and previous-notebook artifacts

Required files are produced by Notebooks 09, 12 and 13. Notebook 14's evaluation summary is optional. If Kaggle finds multiple copies, set an explicit path override before continuing.

In [ ]:
SEARCH_ROOTS = ["/kaggle/input", "/kaggle/working"]
PRUNE_DIRS = {"train_images", "test_images", "train_series", "test_series", "__pycache__", ".git", ".ipynb_checkpoints"}

def find_files(filename):
    hits=[]
    for root in SEARCH_ROOTS:
        if not os.path.exists(root): continue
        for current, dirs, files in os.walk(root):
            dirs[:] = [d for d in dirs if d not in PRUNE_DIRS]
            if filename in files: hits.append(os.path.join(current, filename))
    return sorted(set(hits), key=lambda p:(len(p),p))

def choose_file(filename, preferred=(), required=True, override=None):
    if override:
        if not os.path.isfile(override): raise FileNotFoundError(f"Override path does not exist: {override}")
        return override
    hits=find_files(filename)
    if not hits:
        if required: raise FileNotFoundError(f"Cannot find {filename}. Attach the RSNA dataset / required previous-notebook output using Kaggle Add Input.")
        return None
    for token in preferred:
        subset=[p for p in hits if token.lower() in p.lower()]
        if subset: hits=subset; break
    if len(hits)>1:
        print(f"Multiple copies of {filename}; selecting {hits[0]}")
        for p in hits[:8]: print("  ",p)
    return hits[0]

# Optional exact-path overrides (normally leave as None).
TRAIN_CSV_OVERRIDE = None
EMBEDDING_OVERRIDE = None
METADATA_OVERRIDE = None
TRAIN_CSV_PATH = choose_file("train.csv", ("rsna-knee-abnormality-detection", "rsna"), override=TRAIN_CSV_OVERRIDE)
EMBEDDING_PATH = choose_file("dinov2_vits14_embeddings_float16.npy", ("09-dinov2", "dinov2"), override=EMBEDDING_OVERRIDE)
METADATA_PATH = choose_file("dinov2_vits14_embedding_metadata.csv", ("09-dinov2", "dinov2"), override=METADATA_OVERRIDE)
# Notebook 12 gold-only reference artifacts.
NB12_HINT = "12_proposed_model"
OOF_PATH = choose_file("oof_predictions.csv", (NB12_HINT,))
SLICE_ATT_PATH = choose_file("oof_slice_attention_weights.csv", (NB12_HINT,))
SERIES_ATT_PATH = choose_file("oof_series_attention_weights.csv", (NB12_HINT,))
# Notebook 13 final report-guided model artifacts.
NB13_OOF_PATH = choose_file("oof_predictions_all_modes.csv", ("13_weak_supervision", "13"))
WEAK_MODE = "gold_plus_confidence_weighted_weak"
NB13_CHECKPOINTS = {f: choose_file(f"{WEAK_MODE}_fold_{f}.pth", ("13_weak_supervision", "13")) for f in range(N_FOLDS)}
# Notebook 14 evaluation summary is a supporting artifact.
NB14_METRICS_PATH = choose_file("overall_metrics.csv", ("14_evaluation", "14"), required=False)
for name, path in [("train.csv",TRAIN_CSV_PATH),("Notebook 09 embeddings",EMBEDDING_PATH),("Notebook 09 metadata",METADATA_PATH),("Notebook 12 OOF",OOF_PATH),("Notebook 12 slice attention",SLICE_ATT_PATH),("Notebook 12 series attention",SERIES_ATT_PATH),("Notebook 13 OOF",NB13_OOF_PATH),("Notebook 14 metrics",NB14_METRICS_PATH)]:
    print(f"{name}: {path or 'not attached (optional)'}")
for fold,path in NB13_CHECKPOINTS.items(): print(f"Notebook 13 checkpoint fold {fold}: {path}")

## 3. Load and validate Notebook 12 OOF outputs

Every gold study must have one OOF prediction and one fold assignment. The attention rows are joined to the matching MRI identifiers, not treated as independent patient samples.

In [ ]:
train = pd.read_csv(TRAIN_CSV_PATH)
metadata = pd.read_csv(METADATA_PATH)
oof = pd.read_csv(OOF_PATH)
slice_att = pd.read_csv(SLICE_ATT_PATH)
series_att = pd.read_csv(SERIES_ATT_PATH)
weak_oof_all = pd.read_csv(NB13_OOF_PATH)
nb14_metrics = pd.read_csv(NB14_METRICS_PATH) if NB14_METRICS_PATH else None
embeddings = np.load(EMBEDDING_PATH, mmap_mode="r")

for df, name in [(train,"train.csv"),(metadata,"Notebook 09 metadata"),(oof,"Notebook 12 OOF"),(slice_att,"slice attention"),(series_att,"series attention"),(weak_oof_all,"Notebook 13 OOF")]:
    if "StudyInstanceUID" not in df.columns: raise ValueError(f"{name} lacks StudyInstanceUID")
    df["StudyInstanceUID"] = df["StudyInstanceUID"].astype(str)
for col in LABEL_COLS:
    if col not in train.columns: raise ValueError(f"train.csv missing label column: {col}")
    train[col] = pd.to_numeric(train[col], errors="coerce")
gold = train.dropna(subset=LABEL_COLS).copy()
if gold["StudyInstanceUID"].duplicated().any(): raise ValueError("Duplicate gold StudyInstanceUID rows.")
if len(gold) != 58: raise ValueError(f"Expected 58 gold studies from Notebook 12; found {len(gold)}.")
gold[LABEL_COLS] = gold[LABEL_COLS].astype(int)

required_meta = ["Embedding_Index","StudyInstanceUID","SeriesInstanceUID"]
missing=[c for c in required_meta if c not in metadata.columns]
if missing: raise ValueError(f"Notebook 09 metadata missing columns: {missing}")
if embeddings.ndim != 2 or embeddings.shape[1] != EMBED_DIM: raise ValueError(f"Expected [N,{EMBED_DIM}] embeddings, got {embeddings.shape}")
if len(metadata) != len(embeddings): raise ValueError("Embedding array and metadata lengths differ.")
metadata["Embedding_Index"] = pd.to_numeric(metadata["Embedding_Index"],errors="raise").astype(int)
metadata["StudyInstanceUID"] = metadata["StudyInstanceUID"].astype(str)
metadata["SeriesInstanceUID"] = metadata["SeriesInstanceUID"].astype(str)
metadata = metadata.sort_values("Embedding_Index").reset_index(drop=True)
if not np.array_equal(metadata["Embedding_Index"].to_numpy(), np.arange(len(metadata))): raise ValueError("Embedding_Index must be contiguous from 0.")
if not np.isfinite(np.asarray(embeddings[:min(len(embeddings),1000)],dtype=np.float32)).all(): raise ValueError("Non-finite embedding values detected in initial audit.")

gold_ids=set(gold["StudyInstanceUID"])
if set(metadata["StudyInstanceUID"].unique()) != gold_ids: raise ValueError("Notebook 09 gold embedding studies do not exactly match the structured-label cohort.")
required_oof={"StudyInstanceUID","fold"}
for label in LABEL_COLS: required_oof.update({f"{label}_true",f"{label}_prob"})
missing=sorted(required_oof-set(oof.columns))
if missing: raise ValueError(f"Notebook 12 OOF file missing columns: {missing}")
oof["fold"] = pd.to_numeric(oof["fold"],errors="raise").astype(int)
counts=oof["StudyInstanceUID"].value_counts()
if set(counts.index)!=gold_ids or not (counts==1).all(): raise ValueError("Notebook 12 OOF must contain each gold study exactly once.")
if not set(oof["fold"].unique()).issubset(set(range(N_FOLDS))): raise ValueError("Unexpected fold IDs in OOF file.")
fold_map=oof.set_index("StudyInstanceUID")["fold"].to_dict()
metadata["fold"] = metadata["StudyInstanceUID"].map(fold_map)
if metadata["fold"].isna().any(): raise ValueError("Some embeddings could not be mapped to an OOF fold.")
if not {"slice_attention_weight","Embedding_Index","SeriesInstanceUID"}.issubset(slice_att.columns): raise ValueError("Unexpected Notebook 12 slice-attention schema.")
if not {"series_attention_weight","SeriesInstanceUID"}.issubset(series_att.columns): raise ValueError("Unexpected Notebook 12 series-attention schema.")
for df in (slice_att,series_att): df["StudyInstanceUID"] = df["StudyInstanceUID"].astype(str)
# Select the main confidence-weighted weak-supervision mode and verify it uses the same OOF folds.
if "mode" not in weak_oof_all.columns: raise ValueError("Notebook 13 OOF file must contain a mode column.")
weak_oof = weak_oof_all[weak_oof_all["mode"] == WEAK_MODE].copy()
weak_counts = weak_oof["StudyInstanceUID"].value_counts()
if set(weak_counts.index) != gold_ids or not (weak_counts == 1).all(): raise ValueError("Notebook 13 confidence-weighted OOF must contain every gold study exactly once.")
weak_oof["fold"] = pd.to_numeric(weak_oof["fold"], errors="raise").astype(int)
weak_fold_map = weak_oof.set_index("StudyInstanceUID")["fold"].to_dict()
if weak_fold_map != fold_map: raise ValueError("Notebook 13 and Notebook 12 fold assignments differ.")
for label in LABEL_COLS:
    for suffix in ["true", "prob"]:
        if f"{label}_{suffix}" not in weak_oof.columns: raise ValueError(f"Notebook 13 OOF missing {label}_{suffix}")
if not np.isfinite(weak_oof[[f"{c}_prob" for c in LABEL_COLS]].to_numpy(float)).all(): raise ValueError("Notebook 13 OOF contains non-finite probabilities.")
if nb14_metrics is not None: nb14_metrics.to_csv(OUTPUT_DIR/"15_notebook14_evaluation_reference.csv", index=False)
if len(NB13_CHECKPOINTS)!=N_FOLDS: raise ValueError("All five Notebook 13 confidence-weighted fold checkpoints are required.")
print("Gold studies:",len(gold_ids),"| embeddings:",len(metadata),"| Notebook 12 OOF:",len(oof),"| Notebook 13 selected OOF:",len(weak_oof))
print("Fold counts:",oof[["StudyInstanceUID","fold"]].drop_duplicates()["fold"].value_counts().sort_index().to_dict())

## 4. Retain Notebook 12 gold-only attention as a reference

Notebook 12 saved OOF attention values for its gold-only model. These are kept for comparison. The main explainability outputs for the confidence-weighted model are recomputed from Notebook 13 checkpoints in the next section.

In [ ]:
series_required=["StudyInstanceUID","SeriesInstanceUID","series_attention_weight"]
nb12_series_rank=series_att.copy()
nb12_series_rank["series_attention_weight"] = pd.to_numeric(nb12_series_rank["series_attention_weight"],errors="coerce")
if nb12_series_rank["series_attention_weight"].isna().any(): raise ValueError("Invalid series attention weights.")
nb12_series_rank["within_study_rank"] = nb12_series_rank.groupby("StudyInstanceUID")["series_attention_weight"].rank(method="first",ascending=False).astype(int)
nb12_series_rank=nb12_series_rank.sort_values(["StudyInstanceUID","within_study_rank"])

nb12_slice_rank=slice_att.copy()
nb12_slice_rank["slice_attention_weight"] = pd.to_numeric(nb12_slice_rank["slice_attention_weight"],errors="coerce")
nb12_slice_rank["within_series_rank"] = nb12_slice_rank.groupby(["StudyInstanceUID","SeriesInstanceUID"])["slice_attention_weight"].rank(method="first",ascending=False).astype(int)
nb12_slice_rank=nb12_slice_rank.sort_values(["StudyInstanceUID","SeriesInstanceUID","within_series_rank"])

# Verify attention normalization independently at each hierarchy level.
series_sums=nb12_series_rank.groupby("StudyInstanceUID")["series_attention_weight"].sum()
slice_sums=nb12_slice_rank.groupby(["StudyInstanceUID","SeriesInstanceUID"])["slice_attention_weight"].sum()
if not np.allclose(series_sums.to_numpy(),1.0,atol=1e-4): raise ValueError("Series attention does not sum to one per study.")
if not np.allclose(slice_sums.to_numpy(),1.0,atol=1e-4): raise ValueError("Slice attention does not sum to one per series.")
nb12_series_rank.to_csv(OUTPUT_DIR/"15_nb12_series_attention_ranked.csv",index=False)
nb12_slice_rank.to_csv(OUTPUT_DIR/"15_nb12_slice_attention_ranked.csv",index=False)

def entropy_stats(values):
    w=np.asarray(values,dtype=float); w=w[np.isfinite(w)&(w>0)]
    if len(w)==0: return {"entropy":np.nan,"max_weight":np.nan,"effective_count":np.nan,"n_instances":0}
    entropy=float(-(w*np.log(w)).sum())
    return {"entropy":entropy,"max_weight":float(w.max()),"effective_count":float(np.exp(entropy)),"n_instances":int(len(w))}

series_entropy_rows=[]
for sid,g in nb12_series_rank.groupby("StudyInstanceUID"):
    series_entropy_rows.append({"StudyInstanceUID":sid,**entropy_stats(g["series_attention_weight"])})
slice_entropy_rows=[]
for (sid,ser),g in nb12_slice_rank.groupby(["StudyInstanceUID","SeriesInstanceUID"]):
    slice_entropy_rows.append({"StudyInstanceUID":sid,"SeriesInstanceUID":ser,**entropy_stats(g["slice_attention_weight"])})
series_entropy=pd.DataFrame(series_entropy_rows)
slice_entropy=pd.DataFrame(slice_entropy_rows)
series_entropy.to_csv(OUTPUT_DIR/"15_nb12_series_attention_entropy.csv",index=False)
slice_entropy.to_csv(OUTPUT_DIR/"15_nb12_slice_attention_entropy.csv",index=False)
display(nb12_series_rank.head(10)); display(nb12_slice_rank.head(10))

## 5. Visualize the highest-weighted series and slice groups

The plots are case-level diagnostics. They do not establish which anatomical structure is diseased.

In [ ]:
example_study=nb12_series_rank.groupby("StudyInstanceUID")["series_attention_weight"].max().idxmax()
example_series=nb12_series_rank[nb12_series_rank["StudyInstanceUID"]==example_study].sort_values("series_attention_weight",ascending=False)
fig,ax=plt.subplots(figsize=(10,4))
ax.bar(range(len(example_series)),example_series["series_attention_weight"].to_numpy())
ax.set_xticks(range(len(example_series)))
ax.set_xticklabels(example_series["SeriesInstanceUID"].astype(str).str[-8:],rotation=40,ha="right")
ax.set_ylabel("Series attention weight"); ax.set_title(f"Notebook 12 series attention — study {example_study}")
fig.tight_layout(); fig.savefig(OUTPUT_DIR/"15_nb12_top_series_attention.png",dpi=160,bbox_inches="tight"); plt.show()

example_series_id=example_series.iloc[0]["SeriesInstanceUID"]
example_slices=nb12_slice_rank[(nb12_slice_rank["StudyInstanceUID"]==example_study)&(nb12_slice_rank["SeriesInstanceUID"]==example_series_id)].sort_values("slice_attention_weight",ascending=False).head(20)
fig,ax=plt.subplots(figsize=(11,4))
ax.bar(range(len(example_slices)),example_slices["slice_attention_weight"].to_numpy())
ax.set_xlabel("Top-ranked 2.5D instances (rank order)"); ax.set_ylabel("Slice attention weight")
ax.set_title("Top 2.5D-instance weights in the highest-attention series")
fig.tight_layout(); fig.savefig(OUTPUT_DIR/"15_nb12_top_slice_attention.png",dpi=160,bbox_inches="tight"); plt.show()

fig,axes=plt.subplots(1,2,figsize=(11,4))
axes[0].hist(series_entropy["effective_count"].dropna(),bins=12); axes[0].set_title("Effective number of series"); axes[0].set_xlabel("exp(attention entropy)")
axes[1].hist(slice_entropy["effective_count"].dropna(),bins=15); axes[1].set_title("Effective number of 2.5D instances"); axes[1].set_xlabel("exp(attention entropy)")
fig.tight_layout(); fig.savefig(OUTPUT_DIR/"15_nb12_attention_concentration.png",dpi=160,bbox_inches="tight"); plt.show()
print("Example study:",example_study,"| series:",example_series_id)

## 6. Rebuild the Notebook 13 confidence-weighted hierarchical MIL model

The class below matches the layer names in Notebook 13's `gold_plus_confidence_weighted_weak_fold_*.pth` checkpoints (`norm`, `v`, `u`, `w`). Its optional attention output is used to compute slice- and series-level explanations from the report-guided model itself.

In [ ]:
class GatedAttentionPoolNB13(nn.Module):
    def __init__(self,input_dim=EMBED_DIM,attention_dim=ATTENTION_DIM):
        super().__init__()
        self.norm=nn.LayerNorm(input_dim)
        self.v=nn.Linear(input_dim,attention_dim)
        self.u=nn.Linear(input_dim,attention_dim)
        self.w=nn.Linear(attention_dim,1,bias=False)
    def forward(self,instances):
        if instances.ndim!=2 or instances.shape[0]==0: raise ValueError("Attention requires non-empty [instances,features].")
        x=self.norm(instances)
        scores=self.w(torch.tanh(self.v(x))*torch.sigmoid(self.u(x))).squeeze(-1)
        weights=torch.softmax(scores,dim=0)
        pooled=torch.sum(weights.unsqueeze(-1)*x,dim=0)
        return pooled,weights

class HierarchicalDINOv2MILNB13(nn.Module):
    def __init__(self):
        super().__init__()
        self.slice_attention=GatedAttentionPoolNB13()
        self.series_attention=GatedAttentionPoolNB13()
        self.classifier=nn.Sequential(nn.LayerNorm(EMBED_DIM),nn.Linear(EMBED_DIM,HIDDEN_DIM),nn.GELU(),nn.Dropout(DROPOUT),nn.Linear(HIDDEN_DIM,NUM_CLASSES))
    def forward(self,series_bags,return_attention=False):
        if not series_bags: raise ValueError("Study has no MRI series.")
        series_vectors=[]; slice_weights_all=[]
        for instances in series_bags:
            pooled,weights=self.slice_attention(instances)
            series_vectors.append(pooled); slice_weights_all.append(weights)
        study_vector,series_weights=self.series_attention(torch.stack(series_vectors,dim=0))
        logits=self.classifier(study_vector)
        if return_attention: return logits,slice_weights_all,series_weights
        return logits

def get_study_bags(study_id):
    group=metadata[metadata["StudyInstanceUID"]==str(study_id)]
    result=[]
    for _,sg in group.groupby("SeriesInstanceUID",sort=True):
        idx=sg.sort_values("Embedding_Index")["Embedding_Index"].to_numpy(dtype=np.int64)
        result.append(torch.as_tensor(np.asarray(embeddings[idx],dtype=np.float32),device=DEVICE))
    if not result: raise ValueError(f"No embeddings for {study_id}")
    return result

def load_fold_model(fold):
    path=NB13_CHECKPOINTS[int(fold)]
    try: ckpt=torch.load(path,map_location="cpu",weights_only=False)
    except TypeError: ckpt=torch.load(path,map_location="cpu")
    if ckpt.get("mode")!=WEAK_MODE: raise ValueError(f"Fold {fold}: expected checkpoint mode {WEAK_MODE}, got {ckpt.get('mode')}")
    if ckpt.get("label_cols")!=LABEL_COLS: raise ValueError(f"Fold {fold}: checkpoint label order mismatch.")
    model=HierarchicalDINOv2MILNB13().to(DEVICE)
    model.load_state_dict(ckpt["model_state_dict"],strict=True)
    model.eval()
    return model,ckpt

print("Notebook 13 checkpoint-compatible model ready. All checkpoint loads use strict=True.")

## 7. Recompute confidence-weighted explanations and test perturbation faithfulness on OOF studies

For every gold study, the matching Notebook 13 outer-fold checkpoint is used. The held-out study was excluded from that checkpoint's training. Series and 2.5D-instance attention are therefore computed from the final confidence-weighted model rather than borrowed from the gold-only reference.

In [ ]:
fold_by_study=weak_oof.set_index("StudyInstanceUID")["fold"].astype(int).to_dict()
weighted_prob=weak_oof.set_index("StudyInstanceUID").loc[sorted(gold_ids),[f"{c}_prob" for c in LABEL_COLS]].to_numpy(dtype=float)
weighted_prob_by_study={sid:weighted_prob[i] for i,sid in enumerate(sorted(gold_ids))}
all_slice_records=[]; all_series_records=[]; perturb_rows=[]
optional_meta=["Anatomical_Plane","Fluid_Sensitive","Fat_Suppression","Num_Slices","Prev_Slice_Index","Center_Slice_Index","Next_Slice_Index"]

for fold in range(N_FOLDS):
    model,ckpt=load_fold_model(fold)
    fold_ids=sorted([sid for sid,ff in fold_by_study.items() if ff==fold])
    heldout=set(ckpt.get("outer_heldout_studies",fold_ids))
    if set(fold_ids)!=heldout: raise ValueError(f"Notebook 13 checkpoint fold {fold} held-out IDs do not match OOF metadata.")
    for sid in fold_ids:
        bags=get_study_bags(sid)
        with torch.no_grad():
            logits,slice_weights,series_weights=model(bags,return_attention=True)
            base_prob=torch.sigmoid(logits).cpu().numpy()
        if not np.allclose(base_prob,weighted_prob_by_study[sid],atol=1e-4,rtol=1e-4):
            raise ValueError(f"Fold {fold} checkpoint predictions do not reproduce Notebook 13 OOF row for study {sid}.")
        # Store one row per MRI series with the model's learned series attention.
        sid_meta=metadata[metadata["StudyInstanceUID"]==sid]
        ordered_series=sorted(sid_meta["SeriesInstanceUID"].unique().tolist())
        for series_pos,series_id in enumerate(ordered_series):
            sg=sid_meta[sid_meta["SeriesInstanceUID"]==series_id].sort_values("Embedding_Index")
            srow={"StudyInstanceUID":sid,"SeriesInstanceUID":series_id,"fold":int(fold),
                  "series_attention_weight":float(series_weights[series_pos].detach().cpu()),"Num_2p5D_Instances":int(len(sg))}
            for col in optional_meta:
                if col in sg.columns:
                    vals=sg[col].dropna(); srow[col]=vals.iloc[0] if len(vals) else np.nan
            all_series_records.append(srow)
            sw=slice_weights[series_pos].detach().cpu().numpy()
            if len(sw)!=len(sg): raise ValueError("Slice attention count does not match series metadata rows.")
            for local_i,(_,meta_row) in enumerate(sg.iterrows()):
                rec=meta_row.to_dict(); rec["fold"]=int(fold); rec["slice_attention_weight"]=float(sw[local_i]); all_slice_records.append(rec)

        top_series=int(torch.argmax(series_weights).item())
        top_slice=int(torch.argmax(slice_weights[top_series]).item())
        row={"StudyInstanceUID":sid,"fold":fold,"top_series_position":top_series,"top_slice_position":top_slice,
             "top_series_weight":float(series_weights[top_series].detach().cpu()),
             "top_slice_weight":float(slice_weights[top_series][top_slice].detach().cpu()),
             "base_top_label":LABEL_COLS[int(np.argmax(base_prob))],"base_top_probability":float(np.max(base_prob))}
        if len(bags)>1:
            pert_series=[b for j,b in enumerate(bags) if j!=top_series]
            with torch.no_grad(): p_series=torch.sigmoid(model(pert_series)).cpu().numpy()
            row["series_removal_applied"]=True
            row["mean_abs_probability_change_remove_top_series"]=float(np.mean(np.abs(base_prob-p_series)))
            for j,label in enumerate(LABEL_COLS): row[f"delta_remove_top_series_{label}"]=float(base_prob[j]-p_series[j])
        else:
            row["series_removal_applied"]=False; row["mean_abs_probability_change_remove_top_series"]=np.nan
        if bags[top_series].shape[0]>1:
            pert_slice=[]
            for j,b in enumerate(bags):
                if j==top_series: pert_slice.append(torch.cat([b[:top_slice],b[top_slice+1:]],dim=0))
                else: pert_slice.append(b)
            with torch.no_grad(): p_slice=torch.sigmoid(model(pert_slice)).cpu().numpy()
            row["slice_removal_applied"]=True
            row["mean_abs_probability_change_remove_top_slice"]=float(np.mean(np.abs(base_prob-p_slice)))
            for j,label in enumerate(LABEL_COLS): row[f"delta_remove_top_slice_{label}"]=float(base_prob[j]-p_slice[j])
        else:
            row["slice_removal_applied"]=False; row["mean_abs_probability_change_remove_top_slice"]=np.nan
        perturb_rows.append(row)
    del model,ckpt; gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

series_rank=pd.DataFrame(all_series_records)
slice_rank=pd.DataFrame(all_slice_records)
series_rank["within_study_rank"]=series_rank.groupby("StudyInstanceUID")["series_attention_weight"].rank(method="first",ascending=False).astype(int)
slice_rank["within_series_rank"]=slice_rank.groupby(["StudyInstanceUID","SeriesInstanceUID"])["slice_attention_weight"].rank(method="first",ascending=False).astype(int)
series_rank=series_rank.sort_values(["StudyInstanceUID","within_study_rank"])
slice_rank=slice_rank.sort_values(["StudyInstanceUID","SeriesInstanceUID","within_series_rank"])
series_sums=series_rank.groupby("StudyInstanceUID")["series_attention_weight"].sum()
slice_sums=slice_rank.groupby(["StudyInstanceUID","SeriesInstanceUID"])["slice_attention_weight"].sum()
if not np.allclose(series_sums.to_numpy(),1.0,atol=1e-4): raise ValueError("Notebook 13 series attention does not sum to one per study.")
if not np.allclose(slice_sums.to_numpy(),1.0,atol=1e-4): raise ValueError("Notebook 13 slice attention does not sum to one per series.")
series_rank.to_csv(OUTPUT_DIR/"15_series_attention_ranked.csv",index=False)
slice_rank.to_csv(OUTPUT_DIR/"15_slice_attention_ranked.csv",index=False)

def entropy_stats(values):
    w=np.asarray(values,dtype=float); w=w[np.isfinite(w)&(w>0)]
    if len(w)==0: return {"entropy":np.nan,"max_weight":np.nan,"effective_count":np.nan,"n_instances":0}
    ent=float(-(w*np.log(w)).sum())
    return {"entropy":ent,"max_weight":float(w.max()),"effective_count":float(np.exp(ent)),"n_instances":int(len(w))}
series_entropy=pd.DataFrame([{"StudyInstanceUID":sid,**entropy_stats(g["series_attention_weight"])} for sid,g in series_rank.groupby("StudyInstanceUID")])
slice_entropy=pd.DataFrame([{"StudyInstanceUID":sid,"SeriesInstanceUID":ser,**entropy_stats(g["slice_attention_weight"])} for (sid,ser),g in slice_rank.groupby(["StudyInstanceUID","SeriesInstanceUID"])])
series_entropy.to_csv(OUTPUT_DIR/"15_series_attention_entropy.csv",index=False)
slice_entropy.to_csv(OUTPUT_DIR/"15_slice_attention_entropy.csv",index=False)
perturb_df=pd.DataFrame(perturb_rows)
perturb_df.to_csv(OUTPUT_DIR/"15_perturbation_faithfulness.csv",index=False)
perturb_summary=pd.DataFrame([
    {"perturbation":"remove_top_attention_series","n_testable":int(perturb_df["series_removal_applied"].sum()),"mean_abs_probability_change":perturb_df["mean_abs_probability_change_remove_top_series"].mean(),"median_abs_probability_change":perturb_df["mean_abs_probability_change_remove_top_series"].median()},
    {"perturbation":"remove_top_attention_2p5d_instance","n_testable":int(perturb_df["slice_removal_applied"].sum()),"mean_abs_probability_change":perturb_df["mean_abs_probability_change_remove_top_slice"].mean(),"median_abs_probability_change":perturb_df["mean_abs_probability_change_remove_top_slice"].median()}
])
perturb_summary.to_csv(OUTPUT_DIR/"15_perturbation_summary.csv",index=False)

# Final-model visual diagnostics.
example_study=series_rank.groupby("StudyInstanceUID")["series_attention_weight"].max().idxmax()
example_series=series_rank[series_rank["StudyInstanceUID"]==example_study].sort_values("series_attention_weight",ascending=False)
fig,ax=plt.subplots(figsize=(10,4)); ax.bar(range(len(example_series)),example_series["series_attention_weight"].to_numpy())
ax.set_xticks(range(len(example_series))); ax.set_xticklabels(example_series["SeriesInstanceUID"].astype(str).str[-8:],rotation=40,ha="right")
ax.set_ylabel("Series attention weight"); ax.set_title(f"Notebook 13 confidence-weighted model — study {example_study}")
fig.tight_layout(); fig.savefig(OUTPUT_DIR/"15_top_series_attention.png",dpi=160,bbox_inches="tight"); plt.show()
example_series_id=example_series.iloc[0]["SeriesInstanceUID"]
example_slices=slice_rank[(slice_rank["StudyInstanceUID"]==example_study)&(slice_rank["SeriesInstanceUID"]==example_series_id)].sort_values("slice_attention_weight",ascending=False).head(20)
fig,ax=plt.subplots(figsize=(11,4)); ax.bar(range(len(example_slices)),example_slices["slice_attention_weight"].to_numpy())
ax.set_xlabel("Top-ranked 2.5D instances (rank order)"); ax.set_ylabel("Slice attention weight"); ax.set_title("Top 2.5D-instance weights in the highest-attention series")
fig.tight_layout(); fig.savefig(OUTPUT_DIR/"15_top_slice_attention.png",dpi=160,bbox_inches="tight"); plt.show()
fig,axes=plt.subplots(1,2,figsize=(11,4)); axes[0].hist(series_entropy["effective_count"].dropna(),bins=12); axes[0].set_title("Effective number of series")
axes[1].hist(slice_entropy["effective_count"].dropna(),bins=15); axes[1].set_title("Effective number of 2.5D instances")
fig.tight_layout(); fig.savefig(OUTPUT_DIR/"15_attention_concentration.png",dpi=160,bbox_inches="tight"); plt.show()
print("Notebook 13 weighted-model attention rows:",len(series_rank),len(slice_rank))
display(perturb_summary)

## 8. DINOv2 feature-space attribution (Integrated Gradients)

Compute attribution for the highest-attention cached 2.5D embedding in a small set of OOF studies using the **Notebook 13 confidence-weighted fold model**. This is an attribution over a 384-dimensional feature vector, not an anatomical pixel heatmap.

In [ ]:
ig_rows=[]
selected_ids=sorted(perturb_df.sort_values("base_top_probability",ascending=False)["StudyInstanceUID"].head(MAX_IG_STUDIES).tolist())
for sid in selected_ids:
    fold=int(fold_by_study[sid]); model,ckpt=load_fold_model(fold); bags=get_study_bags(sid)
    with torch.no_grad():
        base_logits,sw_all,rw=model(bags,return_attention=True)
        target=int(torch.argmax(torch.sigmoid(base_logits)).item())
        series_pos=int(torch.argmax(rw).item()); inst_pos=int(torch.argmax(sw_all[series_pos]).item())
    actual=bags[series_pos][inst_pos].detach()
    gradients=[]
    for alpha in torch.linspace(0.0,1.0,IG_STEPS,device=DEVICE):
        variable=(alpha*actual).detach().requires_grad_(True)
        modified=[b.clone() for b in bags]
        modified[series_pos]=modified[series_pos].clone()
        modified[series_pos][inst_pos]=variable
        score=model(modified)[target]
        grad=torch.autograd.grad(score,variable,retain_graph=False,create_graph=False)[0]
        gradients.append(grad.detach())
    avg_grad=torch.stack(gradients,dim=0).mean(dim=0)
    attribution=(actual*avg_grad).detach().abs().cpu().numpy()
    ordered_series_ids=sorted(metadata.loc[metadata["StudyInstanceUID"]==sid,"SeriesInstanceUID"].unique().tolist())
    selected_embedding_index=int(metadata[(metadata["StudyInstanceUID"]==sid)&(metadata["SeriesInstanceUID"]==ordered_series_ids[series_pos])].sort_values("Embedding_Index").iloc[inst_pos]["Embedding_Index"])
    ig_rows.append({"StudyInstanceUID":sid,"fold":fold,"finding":LABEL_COLS[target],"series_position":series_pos,
                    "embedding_index":selected_embedding_index,
                    "instance_position_within_series":inst_pos,"ig_abs_sum":float(attribution.sum()),
                    "ig_abs_mean_per_dimension":float(attribution.mean()),"top_feature_dimension":int(np.argmax(attribution)),
                    "top_feature_attribution":float(np.max(attribution)),"steps":IG_STEPS})
    del model; gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()
ig_df=pd.DataFrame(ig_rows)
ig_df.to_csv(OUTPUT_DIR/"15_feature_space_integrated_gradients.csv",index=False)
display(ig_df)

## 9. Final artifact audit and interpretation

A faithful explanation should be supported by a measurable prediction change when attended content is removed. A large change alone does not establish clinical correctness; inspect the associated MRI images and acquisition metadata before drawing medical conclusions.

In [ ]:
expected=[
    "15_nb12_series_attention_ranked.csv","15_nb12_slice_attention_ranked.csv",
    "15_nb12_series_attention_entropy.csv","15_nb12_slice_attention_entropy.csv",
    "15_series_attention_ranked.csv","15_slice_attention_ranked.csv",
    "15_series_attention_entropy.csv","15_slice_attention_entropy.csv",
    "15_nb12_top_series_attention.png","15_nb12_top_slice_attention.png","15_nb12_attention_concentration.png",
    "15_top_series_attention.png","15_top_slice_attention.png","15_attention_concentration.png",
    "15_perturbation_faithfulness.csv","15_perturbation_summary.csv",
    "15_feature_space_integrated_gradients.csv"
]
if NB14_METRICS_PATH: expected.append("15_notebook14_evaluation_reference.csv")
missing=[f for f in expected if not (OUTPUT_DIR/f).exists()]
audit={"notebook":"15_explainability.ipynb","primary_explanation_model":"Notebook 13 confidence-weighted hierarchical DINOv2 MIL, outer-fold checkpoints",
       "reference_model":"Notebook 12 gold-only OOF attention outputs","notebook14_evaluation_reference_attached":NB14_METRICS_PATH is not None,
       "gold_studies":int(len(gold_ids)),"series_attention_rows":int(len(series_rank)),"slice_attention_rows":int(len(slice_rank)),
       "perturbation_studies":int(len(perturb_df)),"feature_ig_studies":int(len(ig_df)),
       "interpretation_limit":"attention is aggregation weight; Integrated Gradients are feature-space, not pixel-space",
       "missing_artifacts":missing}
with open(OUTPUT_DIR/"run_summary.json","w") as f: json.dump(audit,f,indent=2)
print(json.dumps(audit,indent=2))
if missing: raise RuntimeError("Notebook 15 missing expected outputs: "+", ".join(missing))
print("NOTEBOOK 15 COMPLETED.")